In [2]:
from qiskit import QuantumCircuit, transpile

# Generate benchmark circuits: Clifford-only and Clifford+T

def create_benchmark_clifford_circuits():
    circuits = []
    num_qubits_list = [2,3]
    for n in num_qubits_list:
        # Clifford-only circuit (H, S, CX)
        clifford = QuantumCircuit(n, name=f"clifford_{n}q")
        for i in range(n):
            clifford.h(i)
        for i in range(n-1):
            clifford.cx(i, i+1)
        for i in range(n):
            clifford.s(i)
        clifford.metadata = {"kind": "clifford"}
        circuits.append(clifford)

        # Clifford+T circuit (add T gates)
        clifford_t = QuantumCircuit(n, name=f"clifford_t_{n}q")
        for i in range(n):
            clifford_t.h(i)
        for i in range(n-1):
            clifford_t.cx(i, i+1)
        for i in range(n):
            clifford_t.s(i)
            clifford_t.t(i)
        clifford_t.metadata = {"kind": "clifford_t"}
        circuits.append(clifford_t)

    # Transpile to Clifford+T basis to standardize gates
    transpiled = [transpile(circ, basis_gates=["cx", "h", "s", "t"], optimization_level=3) for circ in circuits]
    for i, circ in enumerate(transpiled):
        circ.metadata = circuits[i].metadata
        circ.name = circuits[i].name
    return transpiled

circuits = create_benchmark_clifford_circuits()
# for idx, qc in enumerate(circuits):
#     display(qc.draw('mpl'))

In [ ]:
# Export transpiled scalable circuits to QASM2 files
import os
import qiskit.qasm2

export_dir = "qasm2_exports"
os.makedirs(export_dir, exist_ok=True)

for idx, qc in enumerate(circuits):
    qasm_str = qiskit.qasm2.dumps(qc)
    # Use circuit metadata to distinguish kinds (e.g., 'clifford', 'clifford_t')
    kind = getattr(qc, 'metadata', {}).get('kind', 'unknown')
    # sanitize kind for filenames
    kind_safe = str(kind).replace(' ', '_')
    filename = os.path.join(export_dir, f"{kind_safe}_{qc.num_qubits}q_{idx+1}.qasm2")
    with open(filename, "w") as f:
        f.write(qasm_str)
    print(f"Exported {filename}")

Exported qasm2_exports/qft_clifford_t_2q_1.qasm2
Exported qasm2_exports/qft_clifford_t_2q_2.qasm2
Exported qasm2_exports/qft_clifford_t_3q_3.qasm2
Exported qasm2_exports/qft_clifford_t_3q_4.qasm2


In [4]:
# Simulate each circuit to get the statevector and benchmark Qiskit
import time
from qiskit_aer import Aer
from qiskit import transpile

simulator = Aer.get_backend('statevector_simulator')

statevectors = []
timings = []

for idx, qc in enumerate(circuits):
    start = time.time()
    tqc = transpile(qc, simulator)
    result = simulator.run(tqc).result()
    statevector = result.get_statevector()
    elapsed = time.time() - start
    statevectors.append(statevector)
    timings.append(elapsed)
    print(f"Simulated circuit {idx+1} ({qc.num_qubits} qubits) in {elapsed:.4f} seconds")

Simulated circuit 1 (2 qubits) in 0.0839 seconds
Simulated circuit 2 (2 qubits) in 0.0608 seconds
Simulated circuit 3 (3 qubits) in 0.0676 seconds
Simulated circuit 4 (3 qubits) in 0.0601 seconds


In [ ]:
print(statevector)

In [4]:
# Display timing results for benchmarking
import pandas as pd

benchmark_data = {
    "Circuit": [f"QFT {qc.num_qubits}q" for qc in circuits],
    "Qubits": [qc.num_qubits for qc in circuits],
    "Simulation Time (s)": timings
}
df = pd.DataFrame(benchmark_data)
display(df)

,Circuit,Qubits,Simulation Time (s)
0,QFT 2q,2,0.046539


## Benchmarking Clifford+T Circuits and Exporting to QASM2
This notebook generates multiple Clifford+T circuits using Qiskit and exports them to QASM2 format for benchmarking.